# Strategy Risk

## Problem Definition

**Question.** What precision and betting frequency would the observed meta-filtered strategy need to reach a target Sharpe ratio?

**Role in the workflow.** Translate final holdout payoff asymmetry into implied precision, frequency, and failure-risk diagnostics.

**Inputs.** Final meta-filtered holdout net returns and backtest statistics.

**Outputs.** `data/backtest_results/strategy_risk.parquet`.

**Why this method.** AFML strategy-risk equations use observed winning/losing outcomes and opportunity frequency rather than a fabricated binary return sample.

**Assumptions.** Target Sharpe is a fixed diagnostic threshold of 1.0, not a claim or tuning objective.

**Handoff.** The final risk interpretation; this ends the research workflow.


## Observed Payoff and Frequency Inputs

- **Purpose.** Derive observed payoff, betting-frequency, and strategy-risk inputs from final strategy returns.
- **Key settings.** `target_sharpe=1.0`; `days_per_year=365.25`; no Gaussian mixture or random return sampling.
- **Data & decision.** Use only fixed meta-filtered holdout events on which the frozen model acts after costs.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.model_backtesting.strategy_risk import (
    implied_betting_frequency,
    implied_precision,
    probability_of_strategy_failure,
)

result_dir = PROJECT_ROOT / "data/backtest_results"
returns = pd.read_parquet(result_dir / "event_strategy_returns.parquet").sort_index()
holdout = returns[returns["partition"].eq("holdout")]
observed = holdout.loc[holdout["meta_action"].eq(1), "meta_filtered_net_return"].astype(float)

winning = observed[observed > 0]
losing = observed[observed <= 0]

elapsed_years = (holdout.index.max() - holdout.index.min()).total_seconds() / (365.25 * 24 * 3600)
frequency = len(observed) / elapsed_years
profit_taking = float(winning.mean())
stop_loss = float(losing.mean())
precision = float((observed > 0).mean())
target_sharpe = 1.0

required_precision = float(implied_precision(stop_loss, profit_taking, frequency, target_sharpe))
required_frequency = float(implied_betting_frequency(stop_loss, profit_taking, precision, target_sharpe))
failure_probability = float(probability_of_strategy_failure(observed.to_numpy(), frequency, target_sharpe))

risk = pd.Series(
    {
        "observed_bets": len(observed),
        "observed_frequency_per_year": frequency,
        "observed_precision": precision,
        "average_winning_net_return": profit_taking,
        "average_losing_net_return": stop_loss,
        "target_sharpe": target_sharpe,
        "required_precision": required_precision,
        "required_frequency": required_frequency,
        "probability_of_strategy_failure": failure_probability,
    },
    name="value",
)
risk.to_frame().to_parquet(result_dir / "strategy_risk.parquet")
display(risk.to_frame())


,value
observed_bets,4.000000
observed_frequency_per_year,24.745240
observed_precision,0.500000
average_winning_net_return,0.009578
average_losing_net_return,-0.001412
target_sharpe,1.000000
required_precision,0.210419
required_frequency,1.811229
probability_of_strategy_failure,0.123366


## Results, Limitations, and Handoff

- **Purpose.** Report the final strategy failure-probability and fragility assessment.
- **Key settings.** No new analytical parameters; the estimate uses a binary-payoff approximation and the fixed holdout sample.
- **Data & decision.** Do not use the diagnostic to retune or deploy the strategy, and produce no downstream artifact beyond the final risk report.